## 1. GridWorld

In [1]:
import sys
import numpy as np
sys.path.append('./src')

from GridWorld import GridWorld

In [2]:
print("=== obstacles=None ===")
env = GridWorld(size=5)
env.reset()
env.render()

=== obstacles=None ===
A . . . .
. . . . .
. . . . .
. . . . .
. . . . G



In [3]:
print("=== Obstacles case ===")
env2 = GridWorld(size=5, obstacles=[(1,1), (2,2), (3,1)])
env2.reset()
env2.render()

=== Obstacles case ===
A . . . .
. X . . .
. . X . .
. X . . .
. . . . G



In [4]:
print("=== Obstacles at start case ===")
try:
    env3 = GridWorld(size=5, obstacles=[(0,0)])
except AssertionError as e:
    print(f"AssertionError: {e}")

try:
    env4 = GridWorld(size=5, obstacles=[(4,4)])
except AssertionError as e:
    print(f"AssertionError: {e}")

=== Obstacles at start case ===
AssertionError: Obstacles at the start or goal position.
AssertionError: Obstacles at the start or goal position.


In [5]:
print("=== Bondary test ===")
env6 = GridWorld(size=5)
env6.reset()
for _ in range(5): 
    state, reward, move, done = env6.step(2)  # left
    print(f"state: {state}, reward: {reward}")

=== Bondary test ===
state: (0, 0), reward: -1
state: (0, 0), reward: -1
state: (0, 0), reward: -1
state: (0, 0), reward: -1
state: (0, 0), reward: -1


In [6]:
print("=== Step test ===")
env5 = GridWorld(size=5, obstacles=[(1,1)])
env5.reset()

print(f"{'state':<12} {'reward':<10} {'action':<10} {'move':<10} {'done':<6}")
print("-" * 50)
for action in [1, 3, 3, 1, 3]: 
    state, reward, move, done = env5.step(action)
    print(f"{str(state):<12} {reward:<10} {action:<10} {move:<10} {str(done):<6}")

print("\nFinal Agent Position:")
env5.render()

=== Step test ===
state        reward     action     move       done  
--------------------------------------------------
(1, 0)       -0.1       1          Down       False 
(1, 0)       -1         3          None       False 
(1, 0)       -1         3          None       False 
(2, 0)       -0.1       1          Down       False 
(2, 1)       -0.1       3          Right      False 

Final Agent Position:
. . . . .
. X . . .
. A . . .
. . . . .
. . . . G



In [7]:
print("=== Icy floor test ===")
env6 = GridWorld(size=5, obstacles=[(1,1), (2,3)], icy_floors=[(0,1), (3,2), (3,3), (1,3)])
env6.reset()

action_list = ['Up', 'Down', 'Left', 'Right']

print(f"{'state':<12} {'reward':<10} {'action':<10} {'move':<10} {'done':<6}")
print("-" * 50)
for action in [1, 2, 3, 1, 3, 0, 1, 3, 3, 2, 3, 2, 3, 0]: 
    state, reward, move, done = env6.step(action)
    print(f"{str(state):<12} {reward:<10} {action:<10} {move:<10} {str(done):<6}")

print("\nFinal Agent Position:")
env6.render()

=== Icy floor test ===
state        reward     action     move       done  
--------------------------------------------------
(1, 0)       -0.1       1          Down       False 
(1, 0)       -1         2          None       False 
(1, 0)       -1         3          None       False 
(2, 0)       -0.1       1          Down       False 
(2, 1)       -0.1       3          Right      False 
(2, 1)       -1         0          None       False 
(3, 1)       -0.1       1          Down       False 
(3, 2)       -0.1       3          Right      False 
(3, 3)       -0.1       3          Right      False 
(3, 3)       -1         2          None       False 
(3, 4)       -0.1       3          Right      False 
(3, 3)       -0.1       2          Left       False 
(3, 3)       -1         3          None       False 
(3, 3)       -1         0          None       False 

Final Agent Position:
. O . . .
. X . O .
. . . X .
. . O A .
. . . . G



# 2. Dynamic Programming
## 2-1. Policy Evaluation

In [8]:
from DP_policyeval import DP_policyeval

env_DP = GridWorld(size=5, obstacles=[(1,1), (2,2), (2,3)], 
                   icy_floors=[(0,1), (1,3), (1,4), (2,0), (3,1), (3,2), (4,1), (4,2)])
states = env_DP.states
actions = env_DP.actions

policy = np.full((len(states), len(actions), 1), 1 / len(actions))

dp_eval = DP_policyeval(env_DP)
V = dp_eval.eval(policy = policy)
print(V.reshape(5, 5))
print(f"\nHighest Value State: {env_DP.index_to_state(np.argmax(V))}, {max(V):.4f}")

[[-40.2391775  -40.26719423 -39.69995008 -39.35226027 -39.4063269 ]
 [-39.61476789 -39.13723038 -39.77130682 -39.22736681 -38.83035013]
 [-38.36873673 -38.07291842 -38.0948069  -37.86718771 -38.11313241]
 [-37.65564855 -37.09318161 -36.66625226 -36.47792398 -36.713621  ]
 [-37.71334034 -37.07258522 -36.59598862 -36.21462686 -36.83116902]]

Highest Value State: (4, 3), -36.2146


## 2-2. Policy Iteration

In [9]:
from DP_policyiter import DP_policyiter

dp_piter = DP_policyiter(env_DP)
policy_p, V_p = dp_piter.forward()

print(f"Optimal Value Function:\n{V_p.reshape(5, 5)}")
print(f"\nOptimal Policy:\n{policy_p.reshape(25, 4)}")

Optimal Value Function:
[[41.06989583 41.35394427 42.22962622 42.75719892 43.29010055]
 [41.58575408 42.22203706 42.48323196 43.01336626 43.82838512]
 [42.10682294 42.7495331  43.44320241 44.17631483 44.72355096]
 [42.7495331  43.28235731 43.98303346 44.72355096 45.27631483]
 [43.05591673 43.59183579 44.42082236 45.27631483 44.72355096]]

Optimal Policy:
[[0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]]


In [10]:
actions = env_DP.actions
states = env_DP.states

env_DP.reset()
agent_pos = env_DP.agent_pos
done = False

print("======= Optimal Policy Steps =======\n")
while not done:
    agent_pos_idx = env_DP.state_to_index(agent_pos)
    action_idx = np.argmax(policy_p[agent_pos_idx])
    state, reward, move, done = env_DP.step(action_idx)
    print(f"{'state:':<6} {str(state):<8} {'reward:':<6} {reward:<6} {'action:':<6} {action_list[action_idx]:<6} {'move:':<4} {move:<6} {'done:':<4} {done}")
    agent_pos = env_DP.agent_pos
    env_DP.render()

======= Optimal Policy Steps =======

state: (1, 0)   reward: -0.1   action: Down   move: Down   done: False
. O . . .
A X . O O
O . X X .
. O O . .
. O O . G

state: (2, 0)   reward: -0.1   action: Down   move: Down   done: False
. O . . .
. X . O O
A . X X .
. O O . .
. O O . G

state: (3, 0)   reward: -0.1   action: Right  move: Down   done: False
. O . . .
. X . O O
O . X X .
A O O . .
. O O . G

state: (3, 1)   reward: -0.1   action: Right  move: Right  done: False
. O . . .
. X . O O
O . X X .
. A O . .
. O O . G

state: (3, 2)   reward: -0.1   action: Right  move: Right  done: False
. O . . .
. X . O O
O . X X .
. O A . .
. O O . G

state: (3, 3)   reward: -0.1   action: Right  move: Right  done: False
. O . . .
. X . O O
O . X X .
. O O A .
. O O . G

state: (4, 3)   reward: -0.1   action: Down   move: Down   done: False
. O . . .
. X . O O
O . X X .
. O O . .
. O O A G

state: (4, 4)   reward: 1      action: Right  move: Right  done: True
. O . . .
. X . O O
O . X X .
. O O . 

## 2-3. Value Iteration

In [11]:
from DP_valueiter import DP_valueiter

dp_viter = DP_valueiter(env_DP)
policy_v, V_v = dp_viter.forward()

print(f"Optimal Value Function:\n{V_v.reshape(5, 5)}")
print(f"\nOptimal Policy:\n{policy_v.reshape(25, 4)}")

Optimal Value Function:
[[41.06991878 41.35396685 42.22964917 42.7572214  43.29012351]
 [41.58577656 42.22206001 42.48325449 43.01338917 43.8284076 ]
 [42.10684589 42.74955559 43.44322537 44.17633725 44.72357398]
 [42.74955559 43.28238026 43.98305594 44.72357398 45.27633725]
 [43.05593964 43.59185831 44.42084532 45.27633725 44.72357398]]

Optimal Policy:
[[0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]]


In [25]:
print(f"Maximum difference between V_p and V_v : {np.max(np.abs(V_p - V_v)):.4f}") 
print(f"Are Policy_p and Policy_v the same? :    {'Yes' if np.allclose(policy_p, policy_v) else 'No'}") 

Maximum difference between V_p and V_v : 0.0000
Are Policy_p and Policy_v the same? :    Yes
